[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Teoria_General_%28Jupyter.gen%29/Teoria_General_%28%28Jupyter.gen.gen%29/PLA-08_Plasmas_Alfven_Waves.ipynb)


# Investigación Avanzada: Plasmas Alfven Waves

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Simulation of 1D ideal MHD Alfven wave propagation
# Equations:
# rho * dv/dt = B0 * dB/dx (linearized momentum)
# dB/dt = B0 * dv/dx (linearized induction)

nx = 200
L = 10.0
dx = L / nx
x = np.linspace(0, L, nx)

B0 = 1.0 # Background magnetic field (T)
rho0 = 1.0 # Plasma density (kg/m^3)
mu0 = 1.0 # Vacuum permeability (set to 1 for normalized units)
vA = B0 / np.sqrt(mu0 * rho0) # Alfven speed

# Time parameters
CFL = 0.5
dt = CFL * dx / vA
nt = 100

# Initial conditions (Gaussian pulse)
v = np.exp(-((x - L/2)**2) / 0.5)
B = -np.exp(-((x - L/2)**2) / 0.5) * np.sqrt(mu0 * rho0)

# Staggered grid (Leapfrog method)
# v at integer time steps, B at half time steps
v_new = np.zeros(nx)
B_new = np.zeros(nx)

history_v = []
history_B = []

for n in range(nt):
    if n % 20 == 0:
        history_v.append(np.copy(v))
        history_B.append(np.copy(B))
    
    # Update B
    for i in range(1, nx-1):
        B_new[i] = B[i] + dt * B0 * (v[i+1] - v[i-1]) / (2 * dx)
    B_new[0] = B_new[1] # BC
    B_new[-1] = B_new[-2]
    B = np.copy(B_new)
    
    # Update v
    for i in range(1, nx-1):
        v_new[i] = v[i] + dt * (B0 / (mu0 * rho0)) * (B[i+1] - B[i-1]) / (2 * dx)
    v_new[0] = v_new[1]
    v_new[-1] = v_new[-2]
    v = np.copy(v_new)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
colors = plt.cm.jet(np.linspace(0, 1, len(history_v)))

for i, (vh, Bh) in enumerate(zip(history_v, history_B)):
    ax1.plot(x, vh, color=colors[i])
    ax2.plot(x, Bh, color=colors[i])

ax1.set_title('Alfven Wave Velocity $v_y$')
ax1.set_ylabel('$v_y$')
ax2.set_title('Alfven Wave Magnetic Field $B_y$')
ax2.set_ylabel('$B_y$')
ax2.set_xlabel('Position x')

sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=nt*dt))
cbar = fig.colorbar(sm, ax=[ax1, ax2], label='Time')

plt.savefig('alfven_wave.png')
